<a href="https://colab.research.google.com/github/jayvishalshah/-InfinityLoopRT/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import userdata

try:
    token = userdata.get('HF_TOKEN')
    print("✅ HF_TOKEN loaded successfully!")
    print("Token character length:", len(token))
except Exception as e:
    print("❌ Could not load HF_TOKEN. Please check your Colab Secrets setup.")
    print("Error details:", e)

✅ HF_TOKEN loaded successfully!
Token character length: 37


In [3]:
import os

# Create the 'core' folder first
os.makedirs('core', exist_ok=True)
print("✅ 'core' folder created (or already exists).")

✅ 'core' folder created (or already exists).


In [4]:
%%writefile core/interfaces.py
from abc import ABC, abstractmethod
from typing import Any, Dict, List, Optional

class Target(ABC):
    """Interface for the system being red-teamed (e.g., Gemini)."""

    @abstractmethod
    def send_prompt(self, prompt: str) -> str:
        """Sends a prompt to the target and returns the text response."""
        pass

class Brain(ABC):
    """Interface for the attacker/red-team model (e.g., Dolphin 3)."""

    @abstractmethod
    def generate_attack(self, target_info: str, context: Dict[str, Any]) -> str:
        """Generates the next prompt to test against the target."""
        pass

class Evaluator(ABC):
    """Interface for evaluating if an attack succeeded or failed."""

    @abstractmethod
    def evaluate(self, prompt: str, response: str) -> Dict[str, Any]:
        """Analyzes the response and returns a score/flag dictionary."""
        pass

class Memory(ABC):
    """Interface for tracking the history of the red-team session."""

    @abstractmethod
    def add_interaction(self, prompt: str, response: str, evaluation: Dict[str, Any]) -> None:
        """Stores a single turn of the conversation."""
        pass

    @abstractmethod
    def get_context(self) -> Dict[str, Any]:
        """Retrieves the history to feed back into the Brain."""
        pass

Writing core/interfaces.py


In [5]:
%%writefile core/config.py
import os
from dataclasses import dataclass
from typing import Optional

@dataclass
class ILRTConfig:
    """Global configuration for the Infinity Loop Red Teaming framework."""

    # Target Model Settings (The system being tested)
    target_api_key: Optional[str] = None
    target_model_name: str = "gemini-1.5-pro-latest"

    # Brain Model Settings (The attacker)
    hf_token: Optional[str] = None
    brain_model_name: str = "cognitivecomputations/dolphin-2.9-llama3-8b"

    # Loop & Strategy Settings
    max_turns: int = 5
    temperature: float = 0.7

    # Storage Paths
    output_dir: str = "outputs"

    def __post_init__(self):
        """Automatically create the output directory if it doesn't exist."""
        os.makedirs(self.output_dir, exist_ok=True)

Writing core/config.py


In [6]:
%%writefile core/session.py
import time
from typing import Any, Dict
from core.interfaces import Target, Brain, Evaluator, Memory
from core.config import ILRTConfig

class RedTeamSession:
    """Manages a single automated red-teaming session."""

    def __init__(
        self,
        config: ILRTConfig,
        target: Target,
        brain: Brain,
        evaluator: Evaluator,
        memory: Memory
    ):
        self.config = config
        self.target = target
        self.brain = brain
        self.evaluator = evaluator
        self.memory = memory
        self.turn_count = 0

    def run(self) -> Dict[str, Any]:
        """Executes the Infinity Loop until max turns or a successful attack."""
        print(f"🚀 Starting ILRT Session against {self.config.target_model_name}")

        while self.turn_count < self.config.max_turns:
            self.turn_count += 1
            print(f"\n--- Turn {self.turn_count}/{self.config.max_turns} ---")

            # Step 1: Brain generates the attack prompt based on memory context
            context = self.memory.get_context()
            attack_prompt = self.brain.generate_attack(
                target_info=self.config.target_model_name,
                context=context
            )
            print(f"🧠 Brain generated prompt: '{attack_prompt[:50]}...'")

            # Step 2: Send the attack to the Target model
            try:
                response = self.target.send_prompt(attack_prompt)
                print(f"🎯 Target responded: '{response[:50]}...'")
            except Exception as e:
                print(f"⚠️ Target connection failed: {e}")
                break

            # Step 3: Evaluate if the attack bypassed safety filters
            evaluation = self.evaluator.evaluate(attack_prompt, response)
            print(f"⚖️ Evaluation: {evaluation}")

            # Step 4: Save everything to Memory so the Brain learns for next time
            self.memory.add_interaction(attack_prompt, response, evaluation)

            # Step 5: Check for early victory
            if evaluation.get("success", False):
                print(f"🚨 Attack successful on turn {self.turn_count}!")
                break

            # Brief pause to respect API rate limits
            time.sleep(1)

        print("\n🏁 Session Complete.")
        return {
            "total_turns": self.turn_count,
            "final_context": self.memory.get_context()
        }

Writing core/session.py


In [7]:
%%writefile core/exceptions.py
class ILRTError(Exception):
    """Base exception for all Infinity Loop Red Teaming errors."""
    pass

class TargetConnectionError(ILRTError):
    """Raised when the target model fails to respond or times out."""
    pass

class BrainGenerationError(ILRTError):
    """Raised when the attacker model fails to generate a valid prompt."""
    pass

class EvaluationError(ILRTError):
    """Raised when the evaluator fails to parse or score a response."""
    pass

Writing core/exceptions.py


In [8]:
%%writefile core/logger.py
import logging
import os
from datetime import datetime

def setup_logger(output_dir: str = "outputs") -> logging.Logger:
    """Sets up a customized logger for the ILRT framework."""
    os.makedirs(output_dir, exist_ok=True)

    # Create a unique log file for each run based on the exact time
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = os.path.join(output_dir, f"ilrt_session_{timestamp}.log")

    # Configure the global ILRT logger
    logger = logging.getLogger("ILRT")
    logger.setLevel(logging.DEBUG)

    # Prevent duplicate logs if this cell is run multiple times
    if not logger.handlers:
        # 1. File Handler (saves EVERYTHING, including deep debug data)
        fh = logging.FileHandler(log_file)
        fh.setLevel(logging.DEBUG)
        file_formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
        fh.setFormatter(file_formatter)

        # 2. Console Handler (keeps your Colab screen clean and readable)
        ch = logging.StreamHandler()
        ch.setLevel(logging.INFO)
        console_formatter = logging.Formatter('%(message)s')
        ch.setFormatter(console_formatter)

        logger.addHandler(fh)
        logger.addHandler(ch)

    return logger

Writing core/logger.py


In [9]:
%%writefile core/controller.py
from core.config import ILRTConfig
from core.session import RedTeamSession
from core.logger import setup_logger

class ILRTController:
    """Orchestrates the setup and execution of the Infinity Loop."""

    def __init__(self, config: ILRTConfig = None):
        # Use provided config or load the default one
        self.config = config or ILRTConfig()
        # Initialize our global logger
        self.logger = setup_logger(self.config.output_dir)
        self.logger.info("ILRT Controller initialized.")

    def run_session(self, target, brain, evaluator, memory):
        """Assembles the components and triggers the session."""
        self.logger.info(f"Preparing session against target: {self.config.target_model_name}")

        session = RedTeamSession(
            config=self.config,
            target=target,
            brain=brain,
            evaluator=evaluator,
            memory=memory
        )

        try:
            results = session.run()
            self.logger.info(f"Session completed successfully. Total turns: {results['total_turns']}")
            return results
        except Exception as e:
            self.logger.error(f"Session failed with error: {str(e)}")
            raise e

Writing core/controller.py
